In [1]:
# Cell 1: Autoreload setup — picks up changes to src/ files without kernel restart
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../src')

import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

print("Setup complete")

Setup complete


## Phase 2 — Feature Engineering

### 1. Building Statistics Consolidation

EDA (Phase 1) found ~20 building/apartment statistic columns clustered in 
`_AVG`/`_MODE`/`_MEDI` triplets, co-missing due to a shared root cause 
(absence of the applicant's building record). This step consolidates each 
triplet to a single `_AVG` column and adds one `BUILDING_INFO_AVAILABLE` flag.

### 0. DAYS_EMPLOYED Fix (moved earlier in pipeline)

Originally explored only in EDA, this must run **before** other Phase 2 steps: 
investigating the OCCUPATION_TYPE missing-category issue (below) revealed 99.96% 
overlap between "not employed" (DAYS_EMPLOYED == 365243) and missing OCCUPATION_TYPE 
— confirming they're largely the same population (pensioners/unemployed with no 
occupation to report). Recreates the EDA fix: IS_NOT_EMPLOYED flag + placeholder 
replaced with NaN.

### 4. Categorical Encoding

... (keep existing target encoding documentation) ...

**Bug found and fixed:** initial target encoding showed 96,391 "unseen categories" 
for OCCUPATION_TYPE when encoding the SAME data it was learned from — impossible 
unless something was wrong. Root cause: pandas groupby() silently drops NaN groups, 
so missing OCCUPATION_TYPE (31.35% of data) never got a learned encoding and fell 
back to a generic global mean. Investigation confirmed missing OCCUPATION_TYPE 
overlaps 99.96% with the not-employed population (IS_NOT_EMPLOYED) — a real, 
informative group, not noise. Fixed by filling NaN with the string 'Missing' 
before grouping in both fit_target_encoding and apply_target_encoding, so it 
receives its own learned, meaningful encoded value.

### credit_card_balance.csv
- 103,558 unique SK_ID_PREV-level applicants in raw file; 86,905 in train (28.26% coverage)
- NAME_CONTRACT_STATUS: 96.31% Active, 3.36% Completed, 0.34% Other — same grouping as POS_CASH
- Drawings/payment columns (~19.5-20% missing) verified as structural zeros: NaN <-> 
  AMT_DRAWINGS_CURRENT==0 / AMT_PAYMENT_TOTAL_CURRENT==0, 1:1 match
- CNT_INSTALMENT_MATURE_CUM / AMT_INST_MIN_REGULARITY (~7.9% missing) verified as 
  loan-life-position missingness (pre-first-instalment-cycle), not missing data
- Raw file unsorted by MONTHS_BALANCE within SK_ID_PREV — explicit sort applied before .last()
- UTILIZATION_RATIO can be negative (overpayment) or >1 (over-limit / post-reduction 
  balance) — both legitimate, not data errors

In [2]:
# Cell 2 (rebuild in correct order)
from feature_engineering import (
    fix_days_employed, consolidate_building_stats, add_ext_source_missing_flags,
    transform_amounts, fit_target_encoding, apply_target_encoding, one_hot_encode_remaining,
    add_ratio_features,
    aggregate_bureau, merge_bureau_features,
    aggregate_previous_application, merge_previous_application_features,
    aggregate_installments, merge_installments_features,
    aggregate_pos_cash, merge_pos_cash_features,
    aggregate_bureau_balance, merge_bureau_balance_features,
    aggregate_credit_card_balance, merge_credit_card_balance_features
)

train_raw = pd.read_csv('../data/raw/home-credit-default-risk/application_train.csv')

train_fe = fix_days_employed(train_raw)
train_fe = consolidate_building_stats(train_fe)
train_fe = add_ext_source_missing_flags(train_fe)
train_fe = transform_amounts(train_fe)

encoding_maps = fit_target_encoding(train_fe)
train_fe = apply_target_encoding(train_fe, encoding_maps)
train_fe = one_hot_encode_remaining(train_fe)

train_fe = add_ratio_features(train_fe)

bureau_raw = pd.read_csv('../data/raw/home-credit-default-risk/bureau.csv')
bureau_agg = aggregate_bureau(bureau_raw)
train_fe = merge_bureau_features(train_fe, bureau_agg)

prev_raw = pd.read_csv('../data/raw/home-credit-default-risk/previous_application.csv')
prev_agg = aggregate_previous_application(prev_raw)
train_fe = merge_previous_application_features(train_fe, prev_agg)

instal_raw = pd.read_csv('../data/raw/home-credit-default-risk/installments_payments.csv')
instal_agg = aggregate_installments(instal_raw)
train_fe = merge_installments_features(train_fe, instal_agg)

pos_cash_raw = pd.read_csv('../data/raw/home-credit-default-risk/POS_CASH_balance.csv')
pos_agg = aggregate_pos_cash(pos_cash_raw)
train_fe = merge_pos_cash_features(train_fe, pos_agg)

bureau_balance_raw = pd.read_csv('../data/raw/home-credit-default-risk/bureau_balance.csv')
bb_agg = aggregate_bureau_balance(bureau_balance_raw, bureau_raw)
train_fe = merge_bureau_balance_features(train_fe, bb_agg)

cc_raw = pd.read_csv('../data/raw/home-credit-default-risk/credit_card_balance.csv')
cc_agg = aggregate_credit_card_balance(cc_raw)
train_fe = merge_credit_card_balance_features(train_fe, cc_agg)

print(f"\nFinal shape: {train_fe.shape}")

IS_NOT_EMPLOYED: 55374 flagged (18.01%)
DAYS_EMPLOYED placeholder replaced with NaN
Found 14 building-stat triplets: ['APARTMENTS', 'BASEMENTAREA', 'YEARS_BEGINEXPLUATATION', 'YEARS_BUILD', 'COMMONAREA', 'ELEVATORS', 'ENTRANCES', 'FLOORSMAX', 'FLOORSMIN', 'LANDAREA', 'LIVINGAPARTMENTS', 'LIVINGAREA', 'NONLIVINGAPARTMENTS', 'NONLIVINGAREA']
Dropped 28 redundant MODE/MEDI columns
Kept 14 _AVG columns + 1 new BUILDING_INFO_AVAILABLE flag
EXT_SOURCE_1_MISSING: 173378 flagged (56.38%)
EXT_SOURCE_3_MISSING: 60965 flagged (19.83%)
AMT_INCOME_TOTAL: capped 3 rows at 10,000,000, log-transformed
AMT_CREDIT: log-transformed (no capping - ceiling confirmed legitimate in EDA)
Learned encoding for ORGANIZATION_TYPE: 58 categories (including 'Missing' if present)
Learned encoding for OCCUPATION_TYPE: 19 categories (including 'Missing' if present)
ORGANIZATION_TYPE: encoded, 0 truly-unseen categories filled with global mean (0.0791)
OCCUPATION_TYPE: encoded, 0 truly-unseen categories filled with globa

**Correction:** the initial flag used only `APARTMENTS_AVG` as a reference column, 
which undercounted availability — some applicants had data in other triplets (e.g. 
`YEARS_BEGINEXPLUATATION_AVG`) even when missing `APARTMENTS_AVG`. Fixed to check 
**any** of the 14 `_AVG` columns via `.any(axis=1)`, which more accurately captures 
"does this applicant have any building record at all."

**Corrected result:** 158,701 applicants (51.6%) have at least partial building 
info vs. 148,810 (48.4%) with none — a near-even split, revised from the earlier 
(undercounted) 156,061/151,450 split.

In [3]:
# Run this immediately after Cell 2, nothing in between
print(f"encoding_maps keys: {list(encoding_maps.keys())}")
print(f"OCCUPATION_TYPE in encoding_maps: {'OCCUPATION_TYPE' in encoding_maps}")
if 'OCCUPATION_TYPE' in encoding_maps:
    print(encoding_maps['OCCUPATION_TYPE'].sort_values())

encoding_maps keys: ['ORGANIZATION_TYPE', 'OCCUPATION_TYPE']
OCCUPATION_TYPE in encoding_maps: True
OCCUPATION_TYPE
Accountants              0.048303
High skill tech staff    0.061599
Managers                 0.062140
Core staff               0.063040
HR staff                 0.063943
IT staff                 0.064639
Missing                  0.065131
Private service staff    0.065988
Medicine staff           0.067002
Secretaries              0.070498
Realty agents            0.078562
Cleaning staff           0.096067
Sales staff              0.096318
Cooking staff            0.104440
Laborers                 0.105788
Security staff           0.107424
Waiters/barmen staff     0.112760
Drivers                  0.113261
Low-skill Laborers       0.171524
Name: TARGET, dtype: float64


In [4]:
# Cell 3: Sanity-check the new flag
print(train_fe['BUILDING_INFO_AVAILABLE'].value_counts())
print(f"\nRetained columns sample:")
print([c for c in train_fe.columns if 'AVG' in c or 'BUILDING' in c])

BUILDING_INFO_AVAILABLE
1    158701
0    148810
Name: count, dtype: int64

Retained columns sample:
['APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BUILD_AVG', 'COMMONAREA_AVG', 'ELEVATORS_AVG', 'ENTRANCES_AVG', 'FLOORSMAX_AVG', 'FLOORSMIN_AVG', 'LANDAREA_AVG', 'LIVINGAPARTMENTS_AVG', 'LIVINGAREA_AVG', 'NONLIVINGAPARTMENTS_AVG', 'NONLIVINGAREA_AVG', 'BUILDING_INFO_AVAILABLE']


In [5]:
# Cell 4: Does building info availability correlate with default risk?
check = train_fe.groupby('BUILDING_INFO_AVAILABLE')['TARGET'].agg(['count', 'mean'])
check.columns = ['count', 'default_rate']
check['default_rate_pct'] = check['default_rate'] * 100
print(check)

                          count  default_rate  default_rate_pct
BUILDING_INFO_AVAILABLE                                        
0                        148810      0.092171          9.217123
1                        158701      0.070000          6.999956


**Finding:** Default rate confirms the same pattern with corrected counts: 9.22% 
(no building info) vs. 7.00% (building info available), both deviating from the 
8.07% baseline in the same direction as before. The relationship is stable across 
both the original and corrected flag logic — strengthens confidence this is a real 
signal (likely tied to housing stability/homeownership) rather than a counting 
artifact. Flag retained as a feature for Phase 3.

### 2. EXT_SOURCE Missingness Flags

EDA found that missingness in `EXT_SOURCE_1` and `EXT_SOURCE_3` carries predictive 
signal beyond their raw values (8.52% vs 7.50% default rate for EXT_SOURCE_1; 
9.31% vs 7.77% for EXT_SOURCE_3). Creating binary flags here, before any imputation, 
preserves this signal as a standalone feature. `EXT_SOURCE_2` excluded — only 0.21% 
missing, too rare to be a useful flag.

In [6]:
# Cell 5: Apply EXT_SOURCE missingness flags
from feature_engineering import add_ext_source_missing_flags

train_fe = add_ext_source_missing_flags(train_fe)
print(f"Shape after: {train_fe.shape}")    

EXT_SOURCE_1_MISSING: 173378 flagged (56.38%)
EXT_SOURCE_3_MISSING: 60965 flagged (19.83%)
Shape after: (307511, 306)


**Result:** Flags created matching EDA findings exactly — `EXT_SOURCE_1_MISSING` 
56.38% (173,378), `EXT_SOURCE_3_MISSING` 19.83% (60,965). Shape: 95 → 97 columns.

### 3. Amount Transforms — Income and Credit

EDA found both `AMT_INCOME_TOTAL` and `AMT_CREDIT` heavily right-skewed, with 
log-transform producing a usable distribution for both. EDA also found and 
validated a single genuine outlier in `AMT_INCOME_TOTAL` (117,000,000 — isolated, 
not part of a legitimate heavy tail) which is capped at 10,000,000 before 
log-transforming. `AMT_CREDIT`'s max (4,050,000) was confirmed to be a legitimate 
loan product ceiling — no capping applied there.

In [7]:
# Cell 6: Apply amount transforms
from feature_engineering import transform_amounts

train_fe = transform_amounts(train_fe)
print(f"Shape after: {train_fe.shape}")

print(train_fe[['AMT_INCOME_TOTAL', 'AMT_INCOME_TOTAL_LOG', 'AMT_CREDIT', 'AMT_CREDIT_LOG']].describe())

AMT_INCOME_TOTAL: capped 0 rows at 10,000,000, log-transformed
AMT_CREDIT: log-transformed (no capping - ceiling confirmed legitimate in EDA)
Shape after: (307511, 306)
       AMT_INCOME_TOTAL  AMT_INCOME_TOTAL_LOG    AMT_CREDIT  AMT_CREDIT_LOG
count      3.075110e+05         307511.000000  3.075110e+05   307511.000000
mean       1.684126e+05             11.909234  5.990260e+05       13.070108
std        1.056929e+05              0.488791  4.024908e+05        0.715193
min        2.565000e+04             10.152338  4.500000e+04       10.714440
25%        1.125000e+05             11.630717  2.700000e+05       12.506181
50%        1.471500e+05             11.899215  5.135310e+05       13.149068
75%        2.025000e+05             12.218500  8.086500e+05       13.603123
max        1.000000e+07             16.118096  4.050000e+06       15.214228


**Result:** 3 rows capped in `AMT_INCOME_TOTAL` (matching EDA exactly), both 
columns log-transformed. Raw values preserved as `_RAW` columns for reference. 
Shape: 97 → 101 columns.

**Note:** hit a `NameError: name 'np' is not defined` on first run — `numpy` 
import was missing from the top of `feature_engineering.py` (notebook-level 
imports don't carry into imported modules). Fixed by adding `import numpy as np` 
directly in the file.

In [8]:
# Run this right after Cell 6 (transform_amounts), before Cell 7
print("OCCUPATION_TYPE present right after Cell 6:", 'OCCUPATION_TYPE' in train_fe.columns)
print("ORGANIZATION_TYPE present right after Cell 6:", 'ORGANIZATION_TYPE' in train_fe.columns)

OCCUPATION_TYPE present right after Cell 6: False
ORGANIZATION_TYPE present right after Cell 6: False


In [9]:
# Cell 8: Check OCCUPATION_TYPE missingness in the pre-encoding dataframe
print(f"Missing OCCUPATION_TYPE in train_raw: {train_raw['OCCUPATION_TYPE'].isnull().sum()}")
print(f"As percentage: {train_raw['OCCUPATION_TYPE'].isnull().sum() / len(train_raw) * 100:.2f}%")

Missing OCCUPATION_TYPE in train_raw: 96391
As percentage: 31.35%


In [10]:
# Cell 9 (corrected): Check overlap using DAYS_EMPLOYED placeholder directly
occ_missing = train_raw['OCCUPATION_TYPE'].isnull()
not_employed = (train_raw['DAYS_EMPLOYED'] == 365243)

overlap = pd.crosstab(occ_missing, not_employed)
overlap.index = ['Occupation present', 'Occupation missing']
overlap.columns = ['Employed', 'Not employed (365243)']
print(overlap)

                    Employed  Not employed (365243)
Occupation present    211118                      2
Occupation missing     41019                  55372


In [11]:
# Diagnostic: what's actually in encoding_maps right now?
print(type(encoding_maps))
print(encoding_maps.keys() if isinstance(encoding_maps, dict) else "not a dict")

<class 'dict'>
dict_keys(['ORGANIZATION_TYPE', 'OCCUPATION_TYPE'])


In [12]:
# Verify state is now consistent
print(f"train_fe shape: {train_fe.shape}")
print(f"encoding_maps keys: {list(encoding_maps.keys())}")
print(f"encoding_maps['OCCUPATION_TYPE'] has {len(encoding_maps.get('OCCUPATION_TYPE', {}))} categories")

train_fe shape: (307511, 306)
encoding_maps keys: ['ORGANIZATION_TYPE', 'OCCUPATION_TYPE']
encoding_maps['OCCUPATION_TYPE'] has 19 categories


In [13]:
# Diagnostic: what's actually in train_fe right now?
print(f"Shape: {train_fe.shape}")
print(f"Has ORGANIZATION_TYPE column: {'ORGANIZATION_TYPE' in train_fe.columns}")
print(f"Has OCCUPATION_TYPE column: {'OCCUPATION_TYPE' in train_fe.columns}")
print(f"Has BUREAU_RECORD_COUNT: {'BUREAU_RECORD_COUNT' in train_fe.columns}")
print(f"Has INSTAL_RECORD_COUNT: {'INSTAL_RECORD_COUNT' in train_fe.columns}")

Shape: (307511, 306)
Has ORGANIZATION_TYPE column: False
Has OCCUPATION_TYPE column: False
Has BUREAU_RECORD_COUNT: True
Has INSTAL_RECORD_COUNT: True


In [14]:
# Cell 10: Check what the 'Missing' category learned for OCCUPATION_TYPE
print(encoding_maps['OCCUPATION_TYPE'].sort_values())

OCCUPATION_TYPE
Accountants              0.048303
High skill tech staff    0.061599
Managers                 0.062140
Core staff               0.063040
HR staff                 0.063943
IT staff                 0.064639
Missing                  0.065131
Private service staff    0.065988
Medicine staff           0.067002
Secretaries              0.070498
Realty agents            0.078562
Cleaning staff           0.096067
Sales staff              0.096318
Cooking staff            0.104440
Laborers                 0.105788
Security staff           0.107424
Waiters/barmen staff     0.112760
Drivers                  0.113261
Low-skill Laborers       0.171524
Name: TARGET, dtype: float64


**Verification:** the learned 'Missing' encoding for OCCUPATION_TYPE = 0.0651 
(6.51% default rate) — placing it in the lower-risk half of the distribution, 
between IT staff (6.46%) and Private service staff (6.60%), consistent with the 
pensioner-dominated population it represents. This confirms the fix was not 
cosmetic: the previous buggy fallback (generic global mean, 0.0852) would have 
meaningfully mis-encoded this group's true (lower) risk for 31% of the dataset. 

Broader pattern: occupation default rates form a clear socioeconomic gradient — 
white-collar/skilled roles cluster at 5-7% (Accountants, Managers, IT staff), 
manual/physical labor roles cluster at 10-17% (Low-skill Laborers, Drivers, 
Waiters) — consistent with the education and region-tier gradients found in EDA.

In [15]:
# Cell 11: Add ratio and interaction features
from feature_engineering import add_ratio_features

train_fe = add_ratio_features(train_fe)
print(f"\nShape after: {train_fe.shape}")

print(train_fe[['CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 'CREDIT_GOODS_RATIO',
                 'ANNUITY_CREDIT_RATIO', 'AGE_YEARS', 'EMPLOYED_YEARS', 
                 'EMPLOYED_AGE_RATIO', 'INCOME_PER_PERSON']].describe())

Created 8 ratio/interaction features
Checking for inf/extreme values:
  ANNUITY_INCOME_RATIO: 0 inf, 12 null
  CREDIT_GOODS_RATIO: 0 inf, 278 null
  ANNUITY_CREDIT_RATIO: 0 inf, 12 null
  EMPLOYED_YEARS: 0 inf, 55374 null
  EMPLOYED_AGE_RATIO: 0 inf, 55374 null
  INCOME_PER_PERSON: 0 inf, 2 null

Shape after: (307511, 306)
       CREDIT_INCOME_RATIO  ANNUITY_INCOME_RATIO  CREDIT_GOODS_RATIO  ANNUITY_CREDIT_RATIO      AGE_YEARS  EMPLOYED_YEARS  EMPLOYED_AGE_RATIO  INCOME_PER_PERSON
count        307511.000000         307499.000000       307233.000000         307499.000000  307511.000000   252137.000000       252137.000000       3.075090e+05
mean              3.957571              0.180930            1.122995              0.053695      43.906900        6.527500            0.156861       9.297770e+04
std               2.689728              0.094574            0.124045              0.022481      11.947950        6.402081            0.133549       7.264940e+04
min               0.056249     

In [16]:
# Cell 12: Investigate the CREDIT_INCOME_RATIO outlier
extreme_ratio = train_fe.nlargest(5, 'CREDIT_INCOME_RATIO')[['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'CREDIT_INCOME_RATIO']]
print(extreme_ratio)

# Check the EMPLOYED_YEARS near-zero minimum
print(f"\nRows with EMPLOYED_YEARS very close to 0:")
print(train_fe[train_fe['EMPLOYED_YEARS'] < 0.01][['EMPLOYED_YEARS']].describe())

        AMT_INCOME_TOTAL  AMT_CREDIT  CREDIT_INCOME_RATIO
20727            25650.0   2173500.0            84.736842
35791            45000.0   2215224.0            49.227200
226137           45000.0   1800000.0            40.000000
255247           58500.0   2146500.0            36.692308
158077           40500.0   1436850.0            35.477778

Rows with EMPLOYED_YEARS very close to 0:
       EMPLOYED_YEARS
count        8.000000
mean         0.004791
std          0.003509
min         -0.000000
25%          0.002053
50%          0.005476
75%          0.008214
max          0.008214


### 5. Ratio and Interaction Features

Created 8 features contextualizing raw amounts relative to the applicant's own 
profile: CREDIT_INCOME_RATIO, ANNUITY_INCOME_RATIO, CREDIT_GOODS_RATIO, 
ANNUITY_CREDIT_RATIO, AGE_YEARS, EMPLOYED_YEARS, EMPLOYED_AGE_RATIO, INCOME_PER_PERSON.

**Null counts confirmed expected:** EMPLOYED_YEARS/EMPLOYED_AGE_RATIO nulls (55,374) 
match IS_NOT_EMPLOYED exactly — correct, since pensioners have no employment duration. 
ANNUITY_*/CREDIT_GOODS_RATIO/INCOME_PER_PERSON nulls are small (≤278) and trace to 
pre-existing missingness in AMT_ANNUITY, AMT_GOODS_PRICE, CNT_FAM_MEMBERS — negligible.

**Investigated CREDIT_INCOME_RATIO max (84.7):** confirmed driven by genuinely low 
income (25,650 — the dataset's actual minimum) paired with a plausible loan amount 
(2,173,500, well below the 4,050,000 product ceiling) — not a data error like the 
earlier AMT_INCOME_TOTAL outlier. This is a real, informative high-risk signal. 
No capping applied; tree-based models handle this skew natively.

**Investigated EMPLOYED_YEARS near-zero minimum:** confirmed floating-point noise 
from DAYS_EMPLOYED values very close to 0 (8 rows, all under 0.0083 years/~3 days) 
— applicants who started employment essentially on application day. Not a data issue.

In [17]:
# Cell 13: Load bureau.csv and inspect structure first
bureau = pd.read_csv('../data/raw/home-credit-default-risk/bureau.csv')
print(f"Shape: {bureau.shape}")
print(f"Unique SK_ID_CURR: {bureau['SK_ID_CURR'].nunique()}")
print(f"Rows per applicant - describe:")
print(bureau.groupby('SK_ID_CURR').size().describe())
print(f"\nColumns:\n{bureau.columns.tolist()}")

Shape: (1716428, 17)
Unique SK_ID_CURR: 305811
Rows per applicant - describe:
count    305811.000000
mean          5.612709
std           4.430354
min           1.000000
25%           2.000000
50%           4.000000
75%           8.000000
max         116.000000
dtype: float64

Columns:
['SK_ID_CURR', 'SK_ID_BUREAU', 'CREDIT_ACTIVE', 'CREDIT_CURRENCY', 'DAYS_CREDIT', 'CREDIT_DAY_OVERDUE', 'DAYS_CREDIT_ENDDATE', 'DAYS_ENDDATE_FACT', 'AMT_CREDIT_MAX_OVERDUE', 'CNT_CREDIT_PROLONG', 'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT', 'AMT_CREDIT_SUM_LIMIT', 'AMT_CREDIT_SUM_OVERDUE', 'CREDIT_TYPE', 'DAYS_CREDIT_UPDATE', 'AMT_ANNUITY']


In [18]:
# Cell 14: Check CREDIT_ACTIVE distribution before designing the aggregation
print(bureau['CREDIT_ACTIVE'].value_counts())

CREDIT_ACTIVE
Closed      1079273
Active       630607
Sold           6527
Bad debt         21
Name: count, dtype: int64


In [19]:
# Cell 16: Diagnose the bureau merge count discrepancy
print(f"Expected applicants without bureau record: {307511 - 305811}")
print(f"Actual count from HAS_BUREAU_RECORD: {(train_fe['HAS_BUREAU_RECORD']==0).sum()}")

# Check if BUREAU_RECORD_COUNT actually has the expected number of nulls
print(f"\nBUREAU_RECORD_COUNT nulls: {train_fe['BUREAU_RECORD_COUNT'].isnull().sum()}")

# Check column name collisions - did 'BUREAU_RECORD_COUNT' get suffixed during merge?
bureau_related_cols = [c for c in train_fe.columns if 'RECORD_COUNT' in c or 'BUREAU_RECORD' in c]
print(f"\nColumns matching 'RECORD_COUNT' or 'BUREAU_RECORD': {bureau_related_cols}")

Expected applicants without bureau record: 1700
Actual count from HAS_BUREAU_RECORD: 44020

BUREAU_RECORD_COUNT nulls: 44020

Columns matching 'RECORD_COUNT' or 'BUREAU_RECORD': ['BUREAU_RECORD_COUNT', 'HAS_BUREAU_RECORD', 'PREV_APP_RECORD_COUNT', 'INSTAL_RECORD_COUNT']


In [20]:
# Cell 17: Check SK_ID_CURR overlap directly between train and bureau_agg
train_ids = set(train_raw['SK_ID_CURR'])
bureau_ids = set(bureau_agg['SK_ID_CURR'])

print(f"train_raw unique IDs: {len(train_ids)}")
print(f"bureau_agg unique IDs: {len(bureau_ids)}")
print(f"IDs in train but NOT in bureau_agg: {len(train_ids - bureau_ids)}")
print(f"IDs in bureau_agg but NOT in train: {len(bureau_ids - train_ids)}")

# Check dtypes - a dtype mismatch (e.g. int64 vs object) would break the merge silently
print(f"\ntrain_raw SK_ID_CURR dtype: {train_raw['SK_ID_CURR'].dtype}")
print(f"bureau_agg SK_ID_CURR dtype: {bureau_agg['SK_ID_CURR'].dtype}")

train_raw unique IDs: 307511
bureau_agg unique IDs: 305811
IDs in train but NOT in bureau_agg: 44020
IDs in bureau_agg but NOT in train: 42320

train_raw SK_ID_CURR dtype: int64
bureau_agg SK_ID_CURR dtype: int64


### 6. Bureau Table Aggregation

Aggregated `bureau.csv` (1,716,428 rows, multiple bureau credit-lines per 
applicant) into one row per `SK_ID_CURR`: CREDIT_ACTIVE and CREDIT_TYPE pivoted 
into count columns (status/type carries distinct risk meaning, not safely 
averaged), numeric fields aggregated via min/max/mean/sum as appropriate.

**Reasoning error caught and corrected:** initially predicted only ~1,700 
applicants (307,511 - 305,811) would have no bureau record, based on the naive 
assumption that all 305,811 unique IDs in `bureau.csv` belong to the training 
set. Direct ID-overlap check revealed `bureau.csv` contains records for **both** 
train and test applicants combined (42,320 of its 305,811 IDs belong to the 
test set, not train). 

**Corrected finding: 44,020 applicants (14.31% of training data) have no 
bureau record at all** — a substantially larger "credit-invisible-within-bureau" 
population than initially assumed, and a stronger empirical anchor for this 
project's underbanked-population framing than the original (incorrect) estimate.

This was a reasoning error, not a code bug — the aggregation and merge logic 
were correct throughout; the mistake was in the train/test composition 
assumption used to sanity-check the result. Important methodological lesson: 
verify ID-set overlaps directly rather than inferring them from row-count 
arithmetic alone.

In [21]:
# Cell 18: Load previous_application.csv and inspect structure
prev_app = pd.read_csv('../data/raw/home-credit-default-risk/previous_application.csv')
print(f"Shape: {prev_app.shape}")
print(f"Unique SK_ID_CURR: {prev_app['SK_ID_CURR'].nunique()}")
print(f"Rows per applicant - describe:")
print(prev_app.groupby('SK_ID_CURR').size().describe())
print(f"\nColumns:\n{prev_app.columns.tolist()}")   

Shape: (1670214, 37)
Unique SK_ID_CURR: 338857
Rows per applicant - describe:
count    338857.000000
mean          4.928964
std           4.220716
min           1.000000
25%           2.000000
50%           4.000000
75%           7.000000
max          77.000000
dtype: float64

Columns:
['SK_ID_PREV', 'SK_ID_CURR', 'NAME_CONTRACT_TYPE', 'AMT_ANNUITY', 'AMT_APPLICATION', 'AMT_CREDIT', 'AMT_DOWN_PAYMENT', 'AMT_GOODS_PRICE', 'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START', 'FLAG_LAST_APPL_PER_CONTRACT', 'NFLAG_LAST_APPL_IN_DAY', 'RATE_DOWN_PAYMENT', 'RATE_INTEREST_PRIMARY', 'RATE_INTEREST_PRIVILEGED', 'NAME_CASH_LOAN_PURPOSE', 'NAME_CONTRACT_STATUS', 'DAYS_DECISION', 'NAME_PAYMENT_TYPE', 'CODE_REJECT_REASON', 'NAME_TYPE_SUITE', 'NAME_CLIENT_TYPE', 'NAME_GOODS_CATEGORY', 'NAME_PORTFOLIO', 'NAME_PRODUCT_TYPE', 'CHANNEL_TYPE', 'SELLERPLACE_AREA', 'NAME_SELLER_INDUSTRY', 'CNT_PAYMENT', 'NAME_YIELD_GROUP', 'PRODUCT_COMBINATION', 'DAYS_FIRST_DRAWING', 'DAYS_FIRST_DUE', 'DAYS_LAST_DUE_1ST

In [22]:
# Cell 19: Check NAME_CONTRACT_STATUS distribution
print(prev_app['NAME_CONTRACT_STATUS'].value_counts())

NAME_CONTRACT_STATUS
Approved        1036781
Canceled         316319
Refused          290678
Unused offer      26436
Name: count, dtype: int64


In [23]:
# Cell 20: Check missingness on the rate columns before deciding whether to include them
for col in ['RATE_INTEREST_PRIMARY', 'RATE_INTEREST_PRIVILEGED', 'RATE_DOWN_PAYMENT']:
    pct = prev_app[col].isnull().sum() / len(prev_app) * 100
    print(f"{col}: {pct:.2f}% missing")

RATE_INTEREST_PRIMARY: 99.64% missing
RATE_INTEREST_PRIVILEGED: 99.64% missing
RATE_DOWN_PAYMENT: 53.64% missing


In [24]:
# Cell 21: Check ID overlap directly first (lesson from bureau.csv)
train_ids = set(train_raw['SK_ID_CURR'])
prev_ids = set(prev_app['SK_ID_CURR'])
print(f"train_raw IDs: {len(train_ids)}")
print(f"prev_app unique IDs: {len(prev_ids)}")
print(f"IDs in train but NOT in prev_app: {len(train_ids - prev_ids)}")
print(f"IDs in prev_app but NOT in train: {len(prev_ids - train_ids)}")

train_raw IDs: 307511
prev_app unique IDs: 338857
IDs in train but NOT in prev_app: 16454
IDs in prev_app but NOT in train: 47800


### 7. Previous Application Table Aggregation

Aggregated `previous_application.csv` (1,670,214 rows) into one row per 
`SK_ID_CURR`. NAME_CONTRACT_STATUS pivoted into count columns (Approved/Refused/
Canceled/Unused offer carry distinct risk meaning); added a directly interpretable 
PREV_REFUSAL_RATE feature. RATE_INTEREST_PRIMARY/RATE_INTEREST_PRIVILEGED excluded 
- confirmed 99.64% missing via direct check, essentially no signal.

**Applied lesson from bureau.csv:** verified SK_ID_CURR overlap directly via set 
operations *before* merging, rather than inferring missing-record counts from row 
totals. This confirmed cleanly: 16,454 applicants (5.35%) have no previous Home 
Credit application — exactly matching the merge result, no discrepancy this time.

**Finding:** 94.65% of applicants have at least one prior Home Credit application 
- a high repeat-customer rate, likely reflecting Home Credit's practice of 
campaigning to its existing customer base. The genuinely "blind" population (no 
bureau record AND no prior Home Credit application) is a specific, identifiable 
subgroup worth examining directly when characterizing this project's target 
underbanked population.

In [25]:
# Cell 23: Load installments_payments.csv and inspect structure
installments = pd.read_csv('../data/raw/home-credit-default-risk/installments_payments.csv')
print(f"Shape: {installments.shape}")
print(f"Unique SK_ID_CURR: {installments['SK_ID_CURR'].nunique()}")
print(f"Rows per applicant - describe:")
print(installments.groupby('SK_ID_CURR').size().describe())
print(f"\nColumns:\n{installments.columns.tolist()}")

Shape: (13605401, 8)
Unique SK_ID_CURR: 339587
Rows per applicant - describe:
count    339587.000000
mean         40.064552
std          41.053343
min           1.000000
25%          12.000000
50%          25.000000
75%          51.000000
max         372.000000
dtype: float64

Columns:
['SK_ID_PREV', 'SK_ID_CURR', 'NUM_INSTALMENT_VERSION', 'NUM_INSTALMENT_NUMBER', 'DAYS_INSTALMENT', 'DAYS_ENTRY_PAYMENT', 'AMT_INSTALMENT', 'AMT_PAYMENT']


In [26]:
# Cell 24: Quick check on the payment timing/amount gap before designing aggregation
sample = installments[['DAYS_INSTALMENT', 'DAYS_ENTRY_PAYMENT', 'AMT_INSTALMENT', 'AMT_PAYMENT']].describe()
print(sample)

# Check for nulls in DAYS_ENTRY_PAYMENT / AMT_PAYMENT specifically (missed payments would show here)
print(f"\nDAYS_ENTRY_PAYMENT nulls: {installments['DAYS_ENTRY_PAYMENT'].isnull().sum()}")
print(f"AMT_PAYMENT nulls: {installments['AMT_PAYMENT'].isnull().sum()}")


       DAYS_INSTALMENT  DAYS_ENTRY_PAYMENT  AMT_INSTALMENT   AMT_PAYMENT
count     1.360540e+07        1.360250e+07    1.360540e+07  1.360250e+07
mean     -1.042270e+03       -1.051114e+03    1.705091e+04  1.723822e+04
std       8.009463e+02        8.005859e+02    5.057025e+04  5.473578e+04
min      -2.922000e+03       -4.921000e+03    0.000000e+00  0.000000e+00
25%      -1.654000e+03       -1.662000e+03    4.226085e+03  3.398265e+03
50%      -8.180000e+02       -8.270000e+02    8.884080e+03  8.125515e+03
75%      -3.610000e+02       -3.700000e+02    1.671021e+04  1.610842e+04
max      -1.000000e+00       -1.000000e+00    3.771488e+06  3.771488e+06

DAYS_ENTRY_PAYMENT nulls: 2905
AMT_PAYMENT nulls: 2905


In [27]:
# Cell 25: Check ID overlap, then aggregate and merge
train_ids = set(train_raw['SK_ID_CURR'])
instal_ids = set(installments['SK_ID_CURR'])
print(f"IDs in train but NOT in installments: {len(train_ids - instal_ids)}")

IDs in train but NOT in installments: 15868


In [28]:
# Diagnostic: check what columns actually came out of the new aggregation
print(instal_agg.columns.tolist())

['SK_ID_CURR', 'INSTAL_DAYS_LATE_MAX', 'INSTAL_DAYS_LATE_MEAN', 'INSTAL_AMT_SHORTFALL_MAX', 'INSTAL_AMT_SHORTFALL_MEAN', 'INSTAL_AMT_SHORTFALL_SUM', 'INSTAL_MISSED_PAYMENT_SUM', 'INSTAL_AMT_INSTALMENT_MAX', 'INSTAL_AMT_INSTALMENT_MEAN', 'INSTAL_AMT_INSTALMENT_SUM', 'INSTAL_AMT_PAYMENT_MAX', 'INSTAL_AMT_PAYMENT_MEAN', 'INSTAL_AMT_PAYMENT_SUM', 'INSTAL_NUM_INSTALMENT_NUMBER_MAX', 'INSTAL_RECORD_COUNT', 'INSTAL_LATE_PAYMENT_RATE']


In [29]:
# Diagnostic: check if train_fe already has installment columns from a previous run
instal_cols_in_train_fe = [c for c in train_fe.columns if 'INSTAL' in c]
print(f"INSTAL-related columns already in train_fe: {instal_cols_in_train_fe}")
print(f"\ntrain_fe shape right now: {train_fe.shape}")

INSTAL-related columns already in train_fe: ['INSTAL_DAYS_LATE_MAX', 'INSTAL_DAYS_LATE_MEAN', 'INSTAL_AMT_SHORTFALL_MAX', 'INSTAL_AMT_SHORTFALL_MEAN', 'INSTAL_AMT_SHORTFALL_SUM', 'INSTAL_MISSED_PAYMENT_SUM', 'INSTAL_AMT_INSTALMENT_MAX', 'INSTAL_AMT_INSTALMENT_MEAN', 'INSTAL_AMT_INSTALMENT_SUM', 'INSTAL_AMT_PAYMENT_MAX', 'INSTAL_AMT_PAYMENT_MEAN', 'INSTAL_AMT_PAYMENT_SUM', 'INSTAL_NUM_INSTALMENT_NUMBER_MAX', 'INSTAL_RECORD_COUNT', 'INSTAL_LATE_PAYMENT_RATE', 'HAS_INSTALLMENT_HISTORY', 'POS_CNT_INSTALMENT_MEAN', 'POS_CNT_INSTALMENT_FUTURE_MEAN', 'POS_CNT_INSTALMENT_FUTURE_LAST']

train_fe shape right now: (307511, 306)


In [30]:
# Final check after clean restart
print(f"Shape: {train_fe.shape}")
print(f"INSTAL_RECORD_COUNT present: {'INSTAL_RECORD_COUNT' in train_fe.columns}")
print(f"No duplicate suffixed columns: {[c for c in train_fe.columns if c.endswith('_x') or c.endswith('_y')]}")

Shape: (307511, 306)
INSTAL_RECORD_COUNT present: True
No duplicate suffixed columns: []


**Process note:** hit the same duplicate-merge error three separate times — 
bureau, previous_application, and installments each had a leftover standalone 
aggregation/merge cell from before Cell 2 was rebuilt to consolidate the full 
pipeline. Re-running any of these cells in isolation merged onto a `train_fe` 
that already contained those columns from Cell 2's run, producing `_x`/`_y` 
suffixed duplicates and breaking downstream references (`BUREAU_RECORD_COUNT`, 
`PREV_APP_RECORD_COUNT`, `INSTAL_RECORD_COUNT`). Not a logic bug in any of the 
aggregation functions themselves — a notebook state hygiene issue. All three 
duplicate cells were identified and deleted; Cell 2 is now the single source 
of truth for the entire feature engineering pipeline (bureau through POS_CASH). 
**Going forward: after any fix to a function in `feature_engineering.py`, 
restart the kernel and re-run the full pipeline from Cell 2, rather than 
re-running the affected cell standalone.**

In [31]:
print(f"train_fe shape: {train_fe.shape}")
print(f"encoding_maps keys: {list(encoding_maps.keys())}")

train_fe shape: (307511, 306)
encoding_maps keys: ['ORGANIZATION_TYPE', 'OCCUPATION_TYPE']


### 8. POS_CASH Table Aggregation

Aggregated `POS_CASH_balance.csv` (10,001,358 rows, monthly POS/cash loan 
balance snapshots) into one row per `SK_ID_CURR`. `NAME_CONTRACT_STATUS` 
grouped into `Active`/`Completed`/`Other` before pivoting into count columns — 
checked the full category distribution first (9 categories total) and found 
only Active (91.5%) and Completed (7.4%) exceed 1%, so the remaining 7 rare 
statuses (Signed, Demand, Returned to the store, Approved, Amortized debt, 
Canceled, XNA) are grouped into `Other` to avoid sparse near-empty columns.

`SK_DPD`/`SK_DPD_DEF` aggregated via mean/max (delinquency is a rare-event 
signal at 2.95%/1.14% of rows respectively, confirmed via direct inspection — 
no placeholder-value trap like `DAYS_EMPLOYED`'s 365243). `CNT_INSTALMENT_FUTURE` 
includes both a mean and a `_LAST` (most recent snapshot) variant, since 
remaining instalments as of the latest record is arguably more decision-relevant 
than the historical average — required an explicit sort by `MONTHS_BALANCE` 
within each loan before taking `.last()`, since the raw file order can't be 
trusted to already be sorted.

In [32]:
# Load POS_CASH_balance.csv
pos_cash_raw = pd.read_csv('../data/raw/home-credit-default-risk/POS_CASH_balance.csv')

print(f"POS_CASH_balance shape: {pos_cash_raw.shape}")
print(f"Unique SK_ID_CURR in POS_CASH: {pos_cash_raw['SK_ID_CURR'].nunique()}")
print(f"Unique SK_ID_PREV in POS_CASH: {pos_cash_raw['SK_ID_PREV'].nunique()}")

# The actual overlap check — don't trust row-count arithmetic
train_ids = set(train_fe['SK_ID_CURR'])
pos_ids = set(pos_cash_raw['SK_ID_CURR'])

overlap = train_ids & pos_ids
train_only = train_ids - pos_ids
pos_only = pos_ids - train_ids

print(f"\nTrain applicants WITH POS_CASH record: {len(overlap)} ({len(overlap)/len(train_ids)*100:.2f}%)")
print(f"Train applicants WITHOUT POS_CASH record: {len(train_only)} ({len(train_only)/len(train_ids)*100:.2f}%)")
print(f"POS_CASH IDs NOT in train (i.e. belong to test set): {len(pos_only)} ({len(pos_only)/len(pos_ids)*100:.2f}%)")

POS_CASH_balance shape: (10001358, 8)
Unique SK_ID_CURR in POS_CASH: 337252
Unique SK_ID_PREV in POS_CASH: 936325

Train applicants WITH POS_CASH record: 289444 (94.12%)
Train applicants WITHOUT POS_CASH record: 18067 (5.88%)
POS_CASH IDs NOT in train (i.e. belong to test set): 47808 (14.18%)


**Result:** 289,444 applicants (94.12%) have at least one POS_CASH record; 
18,067 (5.88%) have none — verified via direct ID-set overlap check before 
merging (`set(train_fe['SK_ID_CURR']) & set(pos_cash_raw['SK_ID_CURR'])`), 
same lesson from the bureau coverage surprise. Shape: 266 → 282.

In [33]:
print(pos_cash_raw['NAME_CONTRACT_STATUS'].value_counts())
print(f"\nTotal unique statuses: {pos_cash_raw['NAME_CONTRACT_STATUS'].nunique()}")

NAME_CONTRACT_STATUS
Active                   9151119
Completed                 744883
Signed                     87260
Demand                      7065
Returned to the store       5461
Approved                    4917
Amortized debt               636
Canceled                      15
XNA                            2
Name: count, dtype: int64

Total unique statuses: 9


In [34]:
print(pos_cash_raw[['SK_DPD', 'SK_DPD_DEF']].describe())
print(f"\nMissing SK_DPD: {pos_cash_raw['SK_DPD'].isna().sum()}")
print(f"Missing SK_DPD_DEF: {pos_cash_raw['SK_DPD_DEF'].isna().sum()}")
print(f"\nRows with SK_DPD > 0: {(pos_cash_raw['SK_DPD'] > 0).sum()} ({(pos_cash_raw['SK_DPD'] > 0).mean()*100:.2f}%)")
print(f"Rows with SK_DPD_DEF > 0: {(pos_cash_raw['SK_DPD_DEF'] > 0).sum()} ({(pos_cash_raw['SK_DPD_DEF'] > 0).mean()*100:.2f}%)")

             SK_DPD    SK_DPD_DEF
count  1.000136e+07  1.000136e+07
mean   1.160693e+01  6.544684e-01
std    1.327140e+02  3.276249e+01
min    0.000000e+00  0.000000e+00
25%    0.000000e+00  0.000000e+00
50%    0.000000e+00  0.000000e+00
75%    0.000000e+00  0.000000e+00
max    4.231000e+03  3.595000e+03

Missing SK_DPD: 0
Missing SK_DPD_DEF: 0

Rows with SK_DPD > 0: 295227 (2.95%)
Rows with SK_DPD_DEF > 0: 113969 (1.14%)


In [35]:
# Check MONTHS_BALANCE range per loan to confirm "last" record = most recent
sample_loan = pos_cash_raw[pos_cash_raw['SK_ID_PREV'] == pos_cash_raw['SK_ID_PREV'].iloc[0]]
print(sample_loan[['SK_ID_PREV', 'MONTHS_BALANCE', 'CNT_INSTALMENT_FUTURE', 'NAME_CONTRACT_STATUS']].sort_values('MONTHS_BALANCE'))

         SK_ID_PREV  MONTHS_BALANCE  CNT_INSTALMENT_FUTURE NAME_CONTRACT_STATUS
2862153     1803195             -34                   48.0               Active
6477531     1803195             -33                   47.0               Active
2865318     1803195             -32                   46.0               Active
0           1803195             -31                   45.0               Active
4794783     1803195             -30                   44.0               Active
8125053     1803195             -29                   43.0               Active
3337839     1803195             -28                   42.0               Active
5564316     1803195             -27                   41.0               Active
5628351     1803195             -26                   40.0               Active
3093748     1803195             -25                   39.0               Active
1826804     1803195             -24                   38.0               Active
4193302     1803195             -23     

In [36]:
bureau_balance_raw = pd.read_csv('../data/raw/home-credit-default-risk/bureau_balance.csv')

print(f"bureau_balance shape: {bureau_balance_raw.shape}")
print(f"Unique SK_ID_BUREAU in bureau_balance: {bureau_balance_raw['SK_ID_BUREAU'].nunique()}")
print(bureau_balance_raw.head(10))
print(bureau_balance_raw.dtypes)

bureau_balance shape: (27299925, 3)
Unique SK_ID_BUREAU in bureau_balance: 817395
   SK_ID_BUREAU  MONTHS_BALANCE STATUS
0       5715448               0      C
1       5715448              -1      C
2       5715448              -2      C
3       5715448              -3      C
4       5715448              -4      C
5       5715448              -5      C
6       5715448              -6      C
7       5715448              -7      C
8       5715448              -8      C
9       5715448              -9      0
SK_ID_BUREAU       int64
MONTHS_BALANCE     int64
STATUS            object
dtype: object


In [37]:
print(bureau_balance_raw['STATUS'].value_counts())
print(f"\nTotal unique STATUS values: {bureau_balance_raw['STATUS'].nunique()}")

STATUS
C    13646993
0     7499507
X     5810482
1      242347
5       62406
2       23419
3        8924
4        5847
Name: count, dtype: int64

Total unique STATUS values: 8


In [38]:
print(bureau_balance_raw['MONTHS_BALANCE'].describe())
sample_id = bureau_balance_raw['SK_ID_BUREAU'].iloc[0]
print(bureau_balance_raw[bureau_balance_raw['SK_ID_BUREAU'] == sample_id].sort_values('MONTHS_BALANCE'))

count    2.729992e+07
mean    -3.074169e+01
std      2.386451e+01
min     -9.600000e+01
25%     -4.600000e+01
50%     -2.500000e+01
75%     -1.100000e+01
max      0.000000e+00
Name: MONTHS_BALANCE, dtype: float64
    SK_ID_BUREAU  MONTHS_BALANCE STATUS
26       5715448             -26      X
25       5715448             -25      X
24       5715448             -24      X
23       5715448             -23      X
22       5715448             -22      X
21       5715448             -21      X
20       5715448             -20      X
19       5715448             -19      0
18       5715448             -18      0
17       5715448             -17      0
16       5715448             -16      0
15       5715448             -15      0
14       5715448             -14      0
13       5715448             -13      X
12       5715448             -12      X
11       5715448             -11      X
10       5715448             -10      0
9        5715448              -9      0
8        5715448           

In [39]:
bureau_ids_in_bureau = set(bureau_raw['SK_ID_BUREAU'])
bureau_ids_in_balance = set(bureau_balance_raw['SK_ID_BUREAU'])

overlap = bureau_ids_in_bureau & bureau_ids_in_balance
bureau_only = bureau_ids_in_bureau - bureau_ids_in_balance
balance_only = bureau_ids_in_balance - bureau_ids_in_bureau

print(f"SK_ID_BUREAU in bureau.csv: {len(bureau_ids_in_bureau)}")
print(f"SK_ID_BUREAU in bureau_balance.csv: {len(bureau_ids_in_balance)}")
print(f"Overlap: {len(overlap)}")
print(f"In bureau.csv but NOT in bureau_balance.csv: {len(bureau_only)} ({len(bureau_only)/len(bureau_ids_in_bureau)*100:.2f}%)")
print(f"In bureau_balance.csv but NOT in bureau.csv: {len(balance_only)}")

SK_ID_BUREAU in bureau.csv: 1716428
SK_ID_BUREAU in bureau_balance.csv: 817395
Overlap: 774354
In bureau.csv but NOT in bureau_balance.csv: 942074 (54.89%)
In bureau_balance.csv but NOT in bureau.csv: 43041


In [40]:
# Direct approach: for each applicant, what fraction of their bureau lines have balance history?
bureau_ids_with_balance = set(bureau_balance_raw['SK_ID_BUREAU'])
bureau_raw['HAS_BALANCE'] = bureau_raw['SK_ID_BUREAU'].isin(bureau_ids_with_balance)

coverage_per_applicant = bureau_raw.groupby('SK_ID_CURR')['HAS_BALANCE'].mean()
print(coverage_per_applicant.describe())
print(f"\nApplicants with 0% of lines covered: {(coverage_per_applicant == 0).sum()} ({(coverage_per_applicant == 0).mean()*100:.2f}%)")
print(f"Applicants with 100% of lines covered: {(coverage_per_applicant == 1).sum()} ({(coverage_per_applicant == 1).mean()*100:.2f}%)")

count    305811.000000
mean          0.439707
std           0.496178
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max           1.000000
Name: HAS_BALANCE, dtype: float64

Applicants with 0% of lines covered: 171269 (56.00%)
Applicants with 100% of lines covered: 134108 (43.85%)


In [41]:
# Check per-applicant balance coverage directly, rather than assuming clustering
bureau_ids_with_balance = set(bureau_balance_raw['SK_ID_BUREAU'])
bureau_raw['HAS_BALANCE'] = bureau_raw['SK_ID_BUREAU'].isin(bureau_ids_with_balance)

coverage_per_applicant = bureau_raw.groupby('SK_ID_CURR')['HAS_BALANCE'].mean()
print(coverage_per_applicant.describe())
print(f"\nApplicants with 0% of lines covered: {(coverage_per_applicant == 0).sum()} ({(coverage_per_applicant == 0).mean()*100:.2f}%)")
print(f"Applicants with 100% of lines covered: {(coverage_per_applicant == 1).sum()} ({(coverage_per_applicant == 1).mean()*100:.2f}%)")

count    305811.000000
mean          0.439707
std           0.496178
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max           1.000000
Name: HAS_BALANCE, dtype: float64

Applicants with 0% of lines covered: 171269 (56.00%)
Applicants with 100% of lines covered: 134108 (43.85%)


### 9. Bureau Balance Table Aggregation

Aggregated `bureau_balance.csv` (27,299,925 monthly status rows, keyed by 
`SK_ID_BUREAU` only) via a two-hop merge: aggregated to one row per 
`SK_ID_BUREAU` first, then joined through `bureau.csv`'s `SK_ID_BUREAU` -> 
`SK_ID_CURR` mapping and re-aggregated to applicant level. STATUS pivoted 
into count-per-category columns (8 categories: `0`-`5` DPD severity buckets, 
`C` closed, `X` unknown), plus a max-DPD-severity and ever-late-rate summary 
per applicant.

**Finding — coverage is bimodal, not gradual:** at the bureau-line level, 
54.89% of lines (942,074 of 1,716,428) have no matching balance history. 
Naively this predicts roughly half of any given applicant's lines should be 
missing coverage. Checking directly per applicant shows otherwise: coverage 
is almost perfectly bimodal — 56.00% of applicants have 0% of their lines 
covered, 43.85% have 100% covered, and only ~0.15% fall in between (25th/75th 
percentiles are exactly 0 and 1). Balance-history reporting is effectively an 
institutional switch, not a per-line coin flip — an applicant's bureau lines 
tend to all come from monthly-reporting institutions or none of them do, 
likely reflecting which specific bureau member institutions hold their loans. 
Combined with the 44,020 applicants with no bureau record at all, this yields 
215,280 applicants (70.01%) with zero `bureau_balance` coverage overall. 
`HAS_BUREAU_BALANCE` flag added; NaN preserved (not imputed) for uncovered 
applicants, consistent with how `HAS_BUREAU_RECORD` and `HAS_POS_CASH_RECORD` 
handle missingness elsewhere in the pipeline. Shape: 282 → 288.

In [42]:
cc_raw = pd.read_csv('../data/raw/home-credit-default-risk/credit_card_balance.csv')

print("Shape:", cc_raw.shape)
print("\n--- ID overlap ---")
print("Unique SK_ID_CURR in credit_card_balance:", cc_raw['SK_ID_CURR'].nunique())
print("Coverage vs train applicants:", 
      cc_raw['SK_ID_CURR'].isin(train_raw['SK_ID_CURR']).sum(), 
      "rows /", len(cc_raw))
print("Applicants in train_raw with a CC record:", 
      train_raw['SK_ID_CURR'].isin(cc_raw['SK_ID_CURR']).sum(), 
      "/", len(train_raw))

print("\n--- NAME_CONTRACT_STATUS cardinality ---")
print(cc_raw['NAME_CONTRACT_STATUS'].value_counts(normalize=True) * 100)

print("\n--- MONTHS_BALANCE sort order ---")
is_sorted_per_loan = cc_raw.sort_values(['SK_ID_PREV', 'MONTHS_BALANCE']).index.equals(
    cc_raw.sort_values(['SK_ID_PREV']).index
)
print("Raw file already sorted by MONTHS_BALANCE within SK_ID_PREV:", is_sorted_per_loan)

print("\n--- Missingness ---")
print(cc_raw.isnull().mean().sort_values(ascending=False).head(15) * 100)

print("\n--- Columns ---")
print(cc_raw.columns.tolist())

Shape: (3840312, 23)

--- ID overlap ---
Unique SK_ID_CURR in credit_card_balance: 103558
Coverage vs train applicants: 3227965 rows / 3840312
Applicants in train_raw with a CC record: 86905 / 307511

--- NAME_CONTRACT_STATUS cardinality ---
NAME_CONTRACT_STATUS
Active           96.305613
Completed         3.356967
Signed            0.287945
Demand            0.035544
Sent proposal     0.013358
Refused           0.000443
Approved          0.000130
Name: proportion, dtype: float64

--- MONTHS_BALANCE sort order ---
Raw file already sorted by MONTHS_BALANCE within SK_ID_PREV: False

--- Missingness ---
AMT_PAYMENT_CURRENT           19.998063
CNT_DRAWINGS_POS_CURRENT      19.524872
AMT_DRAWINGS_ATM_CURRENT      19.524872
CNT_DRAWINGS_ATM_CURRENT      19.524872
AMT_DRAWINGS_POS_CURRENT      19.524872
AMT_DRAWINGS_OTHER_CURRENT    19.524872
CNT_DRAWINGS_OTHER_CURRENT    19.524872
CNT_INSTALMENT_MATURE_CUM      7.948208
AMT_INST_MIN_REGULARITY        7.948208
AMT_DRAWINGS_CURRENT           0

In [43]:
mask = cc_raw['AMT_DRAWINGS_ATM_CURRENT'].isnull()
print("Rows where per-channel drawing is NaN:", mask.sum())
print("Of those, AMT_DRAWINGS_CURRENT == 0:", (cc_raw.loc[mask, 'AMT_DRAWINGS_CURRENT'] == 0).sum())
print("Of those, AMT_DRAWINGS_CURRENT != 0:", (cc_raw.loc[mask, 'AMT_DRAWINGS_CURRENT'] != 0).sum())

# Also check the instalment-side missingness separately, different columns
mask2 = cc_raw['CNT_INSTALMENT_MATURE_CUM'].isnull()
print("\nRows where CNT_INSTALMENT_MATURE_CUM is NaN:", mask2.sum())
print("MONTHS_BALANCE distribution for these rows:")
print(cc_raw.loc[mask2, 'MONTHS_BALANCE'].describe())

Rows where per-channel drawing is NaN: 749816
Of those, AMT_DRAWINGS_CURRENT == 0: 749816
Of those, AMT_DRAWINGS_CURRENT != 0: 0

Rows where CNT_INSTALMENT_MATURE_CUM is NaN: 305236
MONTHS_BALANCE distribution for these rows:
count    305236.000000
mean        -52.544130
std          22.092652
min         -96.000000
25%         -72.000000
50%         -46.000000
75%         -33.000000
max         -21.000000
Name: MONTHS_BALANCE, dtype: float64


In [44]:
# Is this actually about calendar-time months, or position within each loan?
non_null_months = cc_raw.loc[~mask2, 'MONTHS_BALANCE']
print("Non-null MONTHS_BALANCE range:", non_null_months.min(), "to", non_null_months.max())

# Per-loan check: does every SK_ID_PREV have SOME non-null CNT_INSTALMENT_MATURE_CUM?
loans_all_null = cc_raw.groupby('SK_ID_PREV')['CNT_INSTALMENT_MATURE_CUM'].apply(lambda s: s.isnull().all())
print("Loans where CNT_INSTALMENT_MATURE_CUM is NaN for ALL months:", loans_all_null.sum(), "/", loans_all_null.shape[0])

Non-null MONTHS_BALANCE range: -96 to -1
Loans where CNT_INSTALMENT_MATURE_CUM is NaN for ALL months: 0 / 104307


In [45]:
# AMT_PAYMENT_CURRENT: is NaN tied to a specific condition, e.g. no payment due this cycle?
mask3 = cc_raw['AMT_PAYMENT_CURRENT'].isnull()
print("Rows where AMT_PAYMENT_CURRENT is NaN:", mask3.sum())
print("AMT_PAYMENT_TOTAL_CURRENT when AMT_PAYMENT_CURRENT is NaN:")
print(cc_raw.loc[mask3, 'AMT_PAYMENT_TOTAL_CURRENT'].describe())
print("\nOf those NaN rows, AMT_PAYMENT_TOTAL_CURRENT == 0:", 
      (cc_raw.loc[mask3, 'AMT_PAYMENT_TOTAL_CURRENT'] == 0).sum(), "/", mask3.sum())

# AMT_INST_MIN_REGULARITY: same per-loan-null check as CNT_INSTALMENT_MATURE_CUM
mask4 = cc_raw['AMT_INST_MIN_REGULARITY'].isnull()
loans_all_null_reg = cc_raw.groupby('SK_ID_PREV')['AMT_INST_MIN_REGULARITY'].apply(lambda s: s.isnull().all())
print("\nLoans where AMT_INST_MIN_REGULARITY is NaN for ALL months:", loans_all_null_reg.sum(), "/", loans_all_null_reg.shape[0])
print("MONTHS_BALANCE range for AMT_INST_MIN_REGULARITY NaN rows:", 
      cc_raw.loc[mask4, 'MONTHS_BALANCE'].min(), "to", cc_raw.loc[mask4, 'MONTHS_BALANCE'].max())

Rows where AMT_PAYMENT_CURRENT is NaN: 767988
AMT_PAYMENT_TOTAL_CURRENT when AMT_PAYMENT_CURRENT is NaN:
count    767988.0
mean          0.0
std           0.0
min           0.0
25%           0.0
50%           0.0
75%           0.0
max           0.0
Name: AMT_PAYMENT_TOTAL_CURRENT, dtype: float64

Of those NaN rows, AMT_PAYMENT_TOTAL_CURRENT == 0: 767988 / 767988

Loans where AMT_INST_MIN_REGULARITY is NaN for ALL months: 0 / 104307
MONTHS_BALANCE range for AMT_INST_MIN_REGULARITY NaN rows: -96 to -21


In [46]:
cc_agg_test = aggregate_credit_card_balance(cc_raw)

print("Shape:", cc_agg_test.shape)
print("Row count matches unique SK_ID_PREV-level applicants:", 
      cc_agg_test['SK_ID_CURR'].nunique(), "vs expected", cc_raw['SK_ID_CURR'].nunique())

print("\n--- Nulls per column ---")
print(cc_agg_test.isnull().sum())

print("\n--- Utilization ratio sanity ---")
print(cc_agg_test[['CC_UTILIZATION_MEAN', 'CC_UTILIZATION_LAST']].describe())
print("Inf values:", np.isinf(cc_agg_test['CC_UTILIZATION_MEAN']).sum(), 
      np.isinf(cc_agg_test['CC_UTILIZATION_LAST']).sum())

print("\n--- Completed rate sanity ---")
print(cc_agg_test['CC_COMPLETED_RATE'].describe())

print("\n--- Spot check one applicant ---")
sample_id = cc_raw['SK_ID_CURR'].iloc[0]
print(cc_raw[cc_raw['SK_ID_CURR'] == sample_id][['MONTHS_BALANCE', 'AMT_BALANCE', 'AMT_CREDIT_LIMIT_ACTUAL', 'NAME_CONTRACT_STATUS']].sort_values('MONTHS_BALANCE'))
print(cc_agg_test[cc_agg_test['SK_ID_CURR'] == sample_id])

Shape: (103558, 19)
Row count matches unique SK_ID_PREV-level applicants: 103558 vs expected 103558

--- Nulls per column ---
SK_ID_CURR                         0
CC_MONTHS_COUNT                    0
CC_BALANCE_MEAN                    0
CC_BALANCE_LAST                    0
CC_CREDIT_LIMIT_MEAN               0
CC_CREDIT_LIMIT_LAST               0
CC_UTILIZATION_MEAN             1113
CC_UTILIZATION_LAST             1113
CC_DRAWINGS_ATM_SUM                0
CC_DRAWINGS_POS_SUM                0
CC_DRAWINGS_OTHER_SUM              0
CC_PAYMENT_CURRENT_MEAN            0
CC_RECEIVABLE_PRINCIPAL_LAST       0
CC_TOTAL_RECEIVABLE_LAST           0
CC_DPD_MEAN                        0
CC_DPD_MAX                         0
CC_DPD_DEF_MEAN                    0
CC_DPD_DEF_MAX                     0
CC_COMPLETED_RATE                  0
dtype: int64

--- Utilization ratio sanity ---
       CC_UTILIZATION_MEAN  CC_UTILIZATION_LAST
count        102445.000000        102445.000000
mean              0.320573  